# HybridFlow-ITS: Model Training Pipeline

This notebook fine-tunes a modern YOLO model (YOLOv8s / YOLOv11s) on the heterogeneous vehicle dataset (car, threewheel, bus, truck, motorbike, van) and exports the weights to ONNX format for the Rust edge engine.

In [ ]:
# 1. Install Dependencies
!pip install ultralytics onnx onnxruntime

In [ ]:
# 2. Verify GPU Availability
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')


In [ ]:
# 3. Train Model on Mixed Traffic Dataset
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    workers=4,
    name='hybridflow_yolo',
    save=True,
    plots=True
)

In [ ]:
# 4. Evaluate Validation Metrics
metrics = model.val()
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

In [ ]:
# 5. Export to ONNX for Rust Core Engine
success = model.export(format='onnx', imgsz=640, dynamic=False, opset=17)
print(f'Export to ONNX finished: {success}')